## Training dataset, independent dataset split

In [ ]:
# read the features df
features_df = pd.read_csv("features_df.csv")
features_df

In [1]:
# Training dataset, independent dataset split

In [ ]:
# check the logic in notebook 1 analysis and figure
# if the uniprot ID have been used in previous dataset, it should be in the training dataset
# if the protein is augmentated from a uniprot ID which are in the previous dataset, it should be in the training dataset

# balance training dataset and independent dataset (80% training and 20% independent dataset)

In [2]:
import pandas as pd
pred_dataset_name = 'data/bioinformatics_31_14_2276_s1/Supplementary_Tables.xlsx'
finder_dataset_name = "data/NIHMS1881042-supplement-Supplementary_table_dataset_2.xlsx"
curated_dataset_name = "data/20240518final_merge_data_with_high_1433pre_mapped_to_human.csv"

In [3]:
# the data in data/bioinformatics_31_14_2276_s1/Supplementary_Tables.xlsx have 4 tab, combine them into one
file_path = pred_dataset_name
excel_file = pd.ExcelFile(file_path)

# Let's get the sheet names
sheet_names = excel_file.sheet_names

# Create an empty list to hold dataframes
dfs = []

# Loop through the sheet names and read each sheet into a dataframe
for sheet in sheet_names:
    df = pd.read_excel(file_path, sheet_name=sheet)
    # Optionally, you can add a column indicating the sheet name, useful for tracking the data source
    df['Source_Sheet'] = sheet
    dfs.append(df)

# Concatenate all the dataframes
pred_dataset_df = pd.concat(dfs, ignore_index=True)
pred_dataset_df = pred_dataset_df[pred_dataset_df['PMID'] != '*Likely NEG sites ']
pred_dataset_df = pred_dataset_df[["Uniprot ID","Site","Residue"]]
pred_dataset_df["Site"] = pred_dataset_df["Site"].astype(int)
pred_dataset_df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: 'data/bioinformatics_31_14_2276_s1/Supplementary_Tables.xlsx'

In [4]:
# 14-3-3 finder dataset preprocess
finder_dataset_df = pd.read_excel(finder_dataset_name)
finder_dataset_df = finder_dataset_df[["Uniprot ID", 'Site',"Residue"]]
finder_dataset_df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: 'data/NIHMS1881042-supplement-Supplementary_table_dataset_2.xlsx'

In [ ]:
import re
previous_datset_df = pd.concat([pred_dataset_df, finder_dataset_df], ignore_index=True)
previous_datset_df

In [ ]:
# put the uniprot ID into a set
uniprotid_dict = set(previous_datset_df["Uniprot ID"])
uniprotid_dict

In [ ]:
training_dataset_df =  features_df[features_df['uniqueID'].isin(uniprotid_dict)]
training_dataset_df["label"].value_counts()

In [ ]:
remaining_dataset_df =  features_df[~features_df['uniqueID'].isin(uniprotid_dict)]
remaining_dataset_df_positive = remaining_dataset_df[remaining_dataset_df['label']==1]
remaining_dataset_df_negative = remaining_dataset_df[remaining_dataset_df['label']==0]

In [ ]:
# add more data into training_dataset_df
# for positive data, add uniprot and augmentated data to about 1200 positive sample

def sample_data(df_source, column_name, target_length):
    # Initialize an empty DataFrame B
    df_target = pd.DataFrame(columns=df_source.columns)
    
    # Keep track of the unique values already processed to avoid duplication
    processed_values = set()

    while len(df_target) < target_length and not df_source.empty:
        # Randomly sample one row from the source DataFrame
        sampled_row = df_source.sample(1)
        k_value = sampled_row.iloc[0][column_name]

        # Check if this value has already been processed
        if k_value not in processed_values:
            # Get all rows with the same value in the specified column
            rows_with_k = df_source[df_source[column_name] == k_value]

            # Add these rows to the target DataFrame
            df_target = pd.concat([df_target, rows_with_k], ignore_index=True)

            # Update the set of processed values
            processed_values.add(k_value)

            # Drop these rows from the source DataFrame to avoid resampling
            df_source = df_source[df_source[column_name] != k_value]

    # Ensure the target DataFrame doesn't exceed the required row count
#     if len(df_target) > target_length:
#         df_target = df_target.iloc[:target_length]

    return df_target


column_name = 'augmentated from uniprot ID'
target_length = 1200 - len(training_dataset_df[training_dataset_df['label']==1])
df_sampling_training_positive = sample_data(remaining_dataset_df_positive, column_name, target_length)
df_sampling_training_positive

In [ ]:
# for the independent negative data, add to 1200, also make sure the sample following whole distribution
sample_size = 1200
df_labeled_and_augmented_negative_data = pd.read_csv("data/labeled_and_augmented_negative_data.csv")
df_values = df_labeled_and_augmented_negative_data['stratification_label'].value_counts(normalize=True)
df_stratum_sample_size = df_values * sample_size
df_stratum_sample_size_ceil = np.ceil(df_stratum_sample_size).astype(int)
df_stratum_sample_size_ceil

In [ ]:
df_sampling_training_negative = pd.DataFrame() 
for index, value in df_stratum_sample_size_ceil.items():
    if value >0:
        df_tmp = df_labeled_and_augmented_negative_data[df_labeled_and_augmented_negative_data["stratification_label"] == index]
        sampled_df = df_tmp.sample(n=value,random_state=42)
        if len(df_sampling_training_negative) == 0:
            df_sampling_training_negative = sampled_df
        else:
            df_sampling_training_negative = pd.concat([df_sampling_training_negative, sampled_df], axis=0)

In [ ]:
training_dataset_df = pd.concat([training_dataset_df, df_sampling_training_positive, df_sampling_training_negative], axis=0)

In [ ]:
training_dataset_df.shape()

In [ ]:
training_dataset_df["label"].value_counts

In [ ]:
# all the remaining data are below to indepnedent dataset, about 300 positive and 300 negative sample
# sampling from the remaining negative sample to get 300 negative sample
sample_size = 300
df_labeled_and_augmented_negative_data = pd.read_csv("data/labeled_and_augmented_negative_data.csv")
df_values = df_labeled_and_augmented_negative_data['stratification_label'].value_counts(normalize=True)
df_stratum_sample_size = df_values * sample_size
df_stratum_sample_size_ceil = np.ceil(df_stratum_sample_size).astype(int)
df_stratum_sample_size_ceil

In [ ]:
df_sampling_indepnedent_negative = pd.DataFrame()
for index, value in df_stratum_sample_size_ceil.items():
    if value >0:
        df_tmp = df_labeled_and_augmented_negative_data[df_labeled_and_augmented_negative_data["stratification_label"] == index]
        df_tmp = df_tmp[(~df_tmp["uniprot ID"].isin(uniprotid_dict))&(~df_tmp["uniprot ID"].isin(set(training_dataset_df["uniprot id"])))]
        sampled_df = df_tmp.sample(n=value,random_state=42)
        if len(df_sampling_indepnedent_negative) == 0:
            df_sampling_indepnedent_negative = sampled_df
        else:
            df_sampling_indepnedent_negative = pd.concat([df_sampling_indepnedent_negative, sampled_df], axis=0)

In [ ]:
# all the remaining positive data as the 300 positive independent dataset
df_sampling_indepnedent_positive

In [ ]:
independent_dataset_df = pd.concat([df_sampling_indepnedent_positive, df_sampling_indepnedent_negative], axis=0)

### 4.3	Feature selection 

task 36: modeling direct from only PLM embedding, we first compare using embeddings from either ESM2 or PTM-mamba to represent the whole protein or the residue embeddings, 

In [ ]:
import pickle
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

def make_pipeline(classifier):
    """Creates a machine learning pipeline with a Random Forest Classifier.

    Returns:
        Pipeline: A scikit-learn pipeline with a RandomForestClassifier as the final step.
    """
    steps = [("classifier", classifier)]
    pipe = Pipeline(steps)
    return pipe

def train(df, classifier):
    """Trains a Random Forest model using the specified DataFrame.

    Parameters:
        df (pd.DataFrame): The DataFrame containing the features and target variable.

    Returns:
        Pipeline: The trained RandomForest model pipeline.
    """
    # We are predicting the protein labels
    y = df["label"]
    target = "label"
    features = list(df.columns)
    
    features.remove(target)

    # Select all columns starting with 'onehot' for features
    x = df[features]

    # Split data into train and test sets
    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    # Count positive and negative labels in y_train and y_test
    y_train_counts = y_train.value_counts()
    y_test_counts = y_test.value_counts()

    print("Label counts in y_train:")
    print(y_train_counts)

    print("\nLabel counts in y_test:")
    print(y_test_counts)

    trained_model = make_pipeline(classifier)
    trained_model.fit(x_train, y_train)

    # Make predictions
    y_train_pred = trained_model.predict(x_train)
    y_test_pred = trained_model.predict(x_test)

    # Calculate model metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    train_report = classification_report(y_train, y_train_pred, zero_division=1)
    test_report = classification_report(y_test, y_test_pred, zero_division=1)

    # Print metrics in a readable format
    print(f"Model {classifer} Performance Metrics:\n")

    print("Training Data Metrics:")
    print(f"Accuracy: {train_accuracy:.2f}")
    print("Classification Report:")
    print(train_report)  # This prints the report as a table

    print("\nTest Data Metrics:")
    print(f"Accuracy: {test_accuracy:.2f}")
    print("Classification Report:")
    print(test_report)  # This prints the report as a table

    print("Successfully trained the model.")
    return trained_model

In [ ]:
training_dataset_df = pd.read_csv("training_dataset_df.csv")
esm_training_dataset_df = df.filter(regex=r'^(esm_protein_embedding_position|esm_residue_embedding_position|label)')
ptmmamba_training_dataset_df = df.filter(regex=r'^(ptmmamba_residue_embedding_position|ptmmamba_protein_embedding_position|label)')

In [ ]:
plm_feature_df_lst = [esm_training_dataset_df, ptmmamba_training_dataset_df]
plm_name_lst = ["esm", "ptmmamba"]

In [1]:
for name, loaded_df in zip(plm_name_lst, plm_feature_df_lst):
    print(f"Modeling using {name}")
    model = train(loaded_df, RandomForestClassifier())

NameError: name 'plm_name_lst' is not defined

task 37: Then we tested modeling from multi-scale biology features plus onehot encoding for the -7 to +7 motif, the result show that multi-scale biology features also can be used to predict the 14-3-3 binding when compare with only use onehot encoded motif feature. 

In [ ]:
excluded_prefixes = (
    'esm_protein_embedding_position',
    'esm_residue_embedding_position',
    'ptmmamba_residue_embedding_position',
    'ptmmamba_protein_embedding_position'
)
selected_columns = df[[col for col in df.columns if not any(col.startswith(prefix) for prefix in excluded_prefixes)]]

biology_training_dataset_df = training_dataset_df[selected_columns]

In [ ]:
print(f"Modeling using multi-scale biology features")
model = train(biology_training_dataset_df, RandomForestClassifier())

task 38: We then test using both PLM and biology features, the results show that using both PLM and biology features shows improved results compared with using only PLM or multi-scale biology features. 

In [ ]:
print(f"Modeling using both PLM and biology features")
model = train(training_dataset_df, RandomForestClassifier())

task 39: feature selection using mRMR. We mainly test how many features are optimized for our model by using different number of features to fit the model and compare the model performance.

In [ ]:
!pip install mrmr_selection

In [ ]:
from mrmr import mrmr_classif
def mrmr_search_feature_number(df, classifier):
    y = df["label"]
    target = "label"
    features = list(df.columns)
    features.remove(target)
    x = df[features]

    # Split data into train and test sets
    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    feature_number_lst = [50,100,200,500,1000]
    for feature_number in feature_number_lst:
        # using mrmr to get the selected features order
        selected_features_order = mrmr_classif(X=X_train, y=y_train, K=feature_number)
        print(f"selected feature number is {len(selected_features_order)}")
        print(selected_features_order)

        X_train_temp = (X_train).copy()[selected_features]
        X_test_temp = X_test.copy()[selected_features]

        trained_model = make_pipeline(classifier)
        trained_model.fit(X_train_temp, y_train)

        # Make predictions
        y_train_pred = trained_model.predict(X_train_temp)
        y_test_pred = trained_model.predict(X_test_temp)

        # Calculate model metrics
        train_accuracy = accuracy_score(y_train, y_train_pred)
        test_accuracy = accuracy_score(y_test, y_test_pred)
        train_report = classification_report(y_train, y_train_pred, zero_division=1)
        test_report = classification_report(y_test, y_test_pred, zero_division=1)

        # Print metrics in a readable format
        print("Model Performance Metrics:\n")

        print("Training Data Metrics:")
        print(f"Accuracy: {train_accuracy:.2f}")
        print("Classification Report:")
        print(train_report)  # This prints the report as a table

        print("\nTest Data Metrics:")
        print(f"Accuracy: {test_accuracy:.2f}")
        print("Classification Report:")
        print(test_report)  # This prints the report as a table

        print(f"Successfully trained the model with {len(selected_features_order)} feature.")

In [ ]:
mrmr_search_feature_number(training_dataset_df, RandomForestClassifier())

task 40: We also test feature selection using another method Boruta, it shows that XX feature give us the best results.

In [ ]:
# initial parameters
def boruta_feature_selection(df, classifier)
    y = df["label"]
    target = "label"
    features = list(df.columns)
    features.remove(target)
    x = df[features]

    # Split data into train and test sets
    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )


    alpha = 0.01
    perc = 100
    max_iter = 80

    features = X_train.columns
    tree_model = classifier
    X_train_temp = X_train.copy()

    # Boruta search
    feat_selector = BorutaPy(tree_model, 
                             n_estimators='auto', 
                             verbose=2, 
                             random_state=0, 
                             perc = perc, 
                             max_iter = max_iter, 
                             alpha = alpha)

    feat_selector.fit(X_train_temp, y_train)

    # get the feature that will be deleted
    rank_mask = feat_selector.ranking_
    deleted_features = [X_train_temp.columns[i] for i, num in enumerate(rank_mask) if (num != 2 and num !=1)]

    # Convert the result set to a string
    result_str = ', '.join(map(str, set(features)-set(deleted_features)))

    # Open a file in write mode and save the result
    with open('20240517 boruta selected features.txt', 'w') as file:
        file.write(result_str)

    # if I want to control the number of feature to selected
    remaining_feature_number = len(X_train.columns) - len(deleted_features)
    print("remaining feature number: ", remaining_feature_number)
    print("remaining feature", set(features)-set(deleted_features))
    return set(features)-set(deleted_features)

In [ ]:
selected_features = boruta_feature_selection(training_dataset_df)

task 41: feature importance and its contribution were
further analyzed to find which feature was more valuable for
the model performance after feature selection 

In [ ]:
# Prefixes for PLM features
plm_prefixes = (
    'esm_protein_embedding_position',
    'esm_residue_embedding_position',
    'ptmmamba_residue_embedding_position',
    'ptmmamba_protein_embedding_position'
)

# Identify PLM and Biological features
plm_features = [feature for feature in selected_features if feature.startswith(plm_prefixes)]
bio_features = [feature for feature in selected_features if not feature.startswith(plm_prefixes)]

# Counts
plm_count = len(plm_features)
bio_count = len(bio_features)

# Output
print(f"PLM features ({plm_count}): {plm_features}")
print(f"Biological features ({bio_count}): {bio_features}")

print(f"PLM features: {plm_count}")
print(f"Biological features: {bio_count}")

### 4.4	Prediction performance with different classifiers

task 42: To test the validity of the optimal feature set in different classifiers, three common classifiers were used to predict 14-3-3 sites: random forest (RF), XGBoost (XGB), SVM and ANN are used. 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

classifer_lst = [RandomForestClassifier(), XGBClassifier(), SVC(), MLPClassifier()]

for classifer in classifer_lst:
    train(df, classifier)

## 4.5	Model architecture and optimization 

task 43: fine-tuned certain hyperparameters (Table X) of the model

In [ ]:
y = df["label"]
target = "label"
features = list(df.columns)
features.remove(target)
x = df[features]

# Split data into train and test sets
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

steps = [('classifier', XGBClassifier())]
pipe = Pipeline(steps)

In [ ]:
parameters = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__learning_rate': [0.01, 0.05, 0.1],
    'classifier__max_depth': [2,3,4],
    'classifier__gamma': [0, 0.5, 1],
    'classifier__reg_alpha': [0, 0.5, 1],
    'classifier__reg_lambda': [0.5, 1, 5],
    'classifier__base_score': [0.2, 0.5, 1]
}

In [ ]:
scorer = make_scorer(average_precision_score)
model_gsv = GridSearchCV(pipe, parameters, cv = 5, scoring = scorer, n_jobs=-1, verbose=0)
model_gsv = model_gsv.fit(X_train, y_train)
model_gsv.best_params_
best_xgboost_model = model_gsv.best_estimator_

In [ ]:
model_gsv.best_params_

In [ ]:
# Evaluate the best model on the test set
y_train_proba = best_xgboost_model.predict_proba(X_train)[:, 1]
y_train_pred = best_xgboost_model.predict(X_train)
    
model_classification_report = classification_report(y_train, y_train_pred)
    
y_valid_proba = best_xgboost_model.predict_proba(X_valid)[:, 1]
y_valid_pred = best_xgboost_model.predict(X_valid)    
    
model_classification_report_valid = classification_report(y_valid, y_valid_pred)

print("XGBoost Model")
print("Training performance:")
print(f"average_precision_score: {average_precision_score(y_train, y_train_proba):.3e}")
print(f"model_classification_report of traning data:")
print(model_classification_report)
print("-" * 20)
print("Testing performance:")
print(f"average_precision_score: {average_precision_score(y_valid, y_valid_proba):.3e}")
print(f"model_classification_report of valida data:")
print(model_classification_report_valid)
print("-" * 20)

## 4.6 Feature Importance analysis

task 44: we conducted a thorough examination of feature importance using SHAP (SHapley Additive exPlanations) values

In [ ]:
best_model = best_xgboost_model

In [ ]:
select_feature_instance = best_model.named_steps['Drop']

# Accessing the mi_select_cols attribute
feature_names = select_feature_instance.select_cols

feature_importances = best_model.named_steps['classifier'].feature_importances_

# Print or use mi_select_cols as needed
print("Selected features: ", feature_names)

feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
feature_importance_df[:20]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Sorting the DataFrame based on the 'Value' column in descending order
df_sorted = feature_importance_df.sort_values(by='Importance', ascending=True)

# Plotting the data
plt.figure(figsize=(10, 6))  # Set the figure size
plt.barh(df_sorted.iloc[:, 0], df_sorted.iloc[:, 1], color='skyblue')  # Create a horizontal bar plot

# Adding title and labels
plt.title('Feature vs Importance')
plt.ylabel(df_sorted.columns[0])
plt.xlabel(df_sorted.columns[1])

# Display the plot
plt.show()

In [ ]:
# !pip install shap
import shap

random_forest_model = best_model.named_steps['classifier']
# print(type(random_forest_model))
# Create a SHAP explainer

# Create a SHAP explainer
explainer = shap.TreeExplainer(random_forest_model)

# Calculate SHAP values
shap_values = explainer(df_drop[feature_names])


# Visualize feature importance with feature values
shap.plots.beeswarm(shap_values, show=True)

task 45: ablation experiments where we systematically removed features to observe changes in model performance.

## 4.7	Performance evaluation and comparison with existing methods, Validation on independent test sets

task 46: Independent datset test with 14-3-3 pred and 14-3-3 site-finder, compare with our results.